In [5]:
import math
import os
import torch
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings

from torch.optim              import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers             import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from tsfm_public              import TimeSeriesForecastingPipeline 
from tsfm_public              import TimeSeriesPreprocessor
from tsfm_public              import TinyTimeMixerForPrediction
from tsfm_public              import TrackingCallback
from tsfm_public              import count_parameters
from tsfm_public              import get_datasets


from tsfm_public.toolkit.time_series_preprocessor import prepare_data_splits
from sklearn.metrics import precision_score, recall_score, f1_score
import ast
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

from sklearn.metrics import precision_score, recall_score, f1_score



def metrics(actual, prediction):
    a = np.stack(actual).flatten()
    p = np.stack(prediction).flatten()
    mask = ~np.isnan(a) & ~np.isnan(p)
    a, p = a[mask], p[mask]

    mae  = np.mean(np.abs(a - p))
    rmse = np.sqrt(np.mean((a - p)**2))
    mape = np.mean(np.abs((a - p) / (a + 1e-8))) * 100

    actual_diff = np.sign(np.diff(a))
    pred_diff   = np.sign(np.diff(p))
    hit_rate = np.mean(actual_diff == pred_diff)

    actual_up = actual_diff > 0
    pred_up   = pred_diff   > 0
    precision = precision_score(actual_up, pred_up, zero_division=0)
    recall    = recall_score(actual_up, pred_up, zero_division=0)
    f1        = f1_score(actual_up, pred_up, zero_division=0)

    return dict(mae=mae, rmse=rmse, mape=mape,
                hit_rate=hit_rate,
                precision=precision, recall=recall, f1=f1)

device             = "cuda" if torch.cuda.is_available() else "cpu"
load_path          =  "ibm-granite/granite-timeseries-ttm-r2"
os.makedirs('fewshot_results', exist_ok=True)


tickers = ["AAPL",
           "TSLA",
           "XOM",
           "SPY",
           "JNJ",
           "AMD",
           "PG"
          ]

event_titles = ["AAPL – Crash & Rebound (2020-03-10)",
                "TSLA – High-Beta Cooling (2021-01-15)",
                "XOM – Oil Cycle Peak (2022-06-01)",
                "SPY – Drawdown Chop (2022-09-15)",
                "JNJ – Low-Volatility Stretch (2019-08-01)",
                "AMD – Tech Selloff (2018-10-10)",
                "PG – Macro-Irrelevant Calm (2015-06-15)"
                ]

starts = ["2019-09-02",
          "2020-07-01",
          "2021-10-01",
          "2022-01-03",
          "2018-11-01",
          "2018-04-02",
          "2014-12-01"
         ]

ends   = ["2020-06-01",
          "2021-06-30",
          "2022-09-30",
          "2022-12-30",
          "2020-01-01",
          "2019-03-29",
          "2016-01-01"
         ]

column_specifiers = {
        "timestamp_column": "date",
        "id_columns": [],
        "target_columns": ["close"],
        "control_columns": [] 
        } 


def create_test_sets():
    test_sets = []
    for ticker in tickers:
        df = pd.read_csv(f"./processed_data/{ticker}.csv", parse_dates=["date"])
        _, _, test_df = prepare_data_splits(df,context_length=512,split_config={"train": 0.6,"test": 0.4})
        test_sets.append(test_df)
    return test_sets

def predict_series( test_sets, pipeline, _type, event_title):
    for test_df,ticker,start,end in zip(test_sets,tickers,starts,ends):
        forecast = pipeline(test_df)
        forecast["date"]     = pd.to_datetime(forecast["date"])
        forecast["y_true"]   = forecast["close"].str[0]
        forecast["y_pred"]   = forecast["close_prediction"].str[0]
        forecast["residual"] = forecast["y_true"] - forecast["y_pred"]
        output_path          = os.path.join('fewshot_results', f'{_type}_predict_{ticker}.csv')
        forecast.to_csv(output_path, index=False)

        df = pd.read_csv(f"./processed_data/{ticker}.csv", parse_dates=["date"])
        forecast         = pipeline(df)
        start            = pd.to_datetime(start)
        end              = pd.to_datetime(end)
        mask             = (forecast["date"] >= start) & (forecast["date"] <= end)
        event            = forecast.loc[mask]
        event["y_true"]  = event["close"].str[0]
        event["y_pred"]  = event["close_prediction"].str[0]
        output_path      = os.path.join('fewshot_results',  f'{_type}_predict_{ticker}_event.csv')
        event.to_csv(output_path, index=False)


test_sets = create_test_sets()
idx = 0

for ticker,start,end,event_title in zip(tickers,starts,ends,event_titles):
    
    df = pd.read_csv(f"./processed_data/{ticker}.csv", parse_dates=["date"])

    
    preprocessor = TimeSeriesPreprocessor(
        **column_specifiers,
        context_length     = 512,
        prediction_length  = 96,
        scaling            = True,
        encode_categorical = False,
        scaler_type        = "standard",
    )
    
    preprocessor.train(df)

    model = TinyTimeMixerForPrediction.from_pretrained(
        load_path , 
        num_input_channels             = preprocessor.num_input_channels,
        prediction_channel_indices     = preprocessor.prediction_channel_indices,
        exogenous_channel_indices      = preprocessor.exogenous_channel_indices,
        fcm_use_mixer                  = False,
        enable_forecast_channel_mixing = False,
        decoder_mode                   = "mix_channel",
    )

    for param in model.backbone.parameters():
        param.requires_grad = False

    train_df, valid_df, test_df = prepare_data_splits(
        df,
        context_length=512,
        split_config={"train": 0.6,"test": 0.4}
    )

    train_set, valid_set, test_set = get_datasets(
        preprocessor,
        df,
        {"train": 0.6,"test": 0.4},
        fewshot_fraction    = 0.7,
        fewshot_location    = "first",
        use_frequency_token = model.config.resolution_prefix_tuning,
    )
    
    learning_rate  = 0.0004
    num_epochs     = 5
    patience       = 10
    batch_size     = 64

    args = TrainingArguments(
        output_dir                  = os.path.join('fewshot_results', "output"),
        overwrite_output_dir        = True,
        learning_rate               = learning_rate,
        num_train_epochs            = num_epochs,
        do_eval                     = True,
        eval_strategy               = "epoch",
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        dataloader_num_workers      = 4,
        report_to                   = None,
        save_strategy               = "epoch",
        logging_strategy            = "epoch",
        save_total_limit            = 1,
        logging_dir                 = os.path.join('fewshot_results', "logs"),  
        load_best_model_at_end      = True,  
        metric_for_best_model       = "eval_loss",  
        greater_is_better           = False,  
        use_cpu                     = device != "cuda",
    )

    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=patience,
        early_stopping_threshold=0.00001, 
        )
    
    tracking_callback = TrackingCallback()

    optimizer = AdamW(model.parameters(), lr=learning_rate,weight_decay=0.01)
    scheduler = OneCycleLR(optimizer, learning_rate, epochs=num_epochs, steps_per_epoch=math.ceil(len(train_set) / (batch_size)),)

    trainer = Trainer(
        model         = model,
        args          = args,
        train_dataset = train_set,
        eval_dataset  = valid_set,
        callbacks     = [early_stopping_callback, tracking_callback],
        optimizers    = (optimizer, scheduler),
    )
    
    trainer.train()

    pipeline = TimeSeriesForecastingPipeline(
        model,
        device            = device, 
        feature_extractor = preprocessor,
        batch_size        = batch_size,
    )

    
    predict_series(
        test_sets,
        pipeline,
        f'finetune_on_{ticker}',
        event_title
    )

results = []
for ticker in tickers:
    path = os.path.join('fewshot_results', f'finetune_on_{ticker}_predict_{ticker}.csv')
    df = pd.read_csv(path)
    df = df.dropna(subset=["y_true", "y_pred"])
    m = metrics(df["y_true"].values, df["y_pred"].values)
    m["ticker"] = ticker
    results.append(m)        

    fewshot_df = pd.DataFrame(results).set_index("ticker")
    output_path = os.path.join("fewshot_results", "finetune_itself.csv")
    fewshot_df.to_csv(output_path)
print('\n\n','--------FINETUNE ITSELF--------')
print(fewshot_df)



for from_ticker in tickers:
    results = []
    for to_ticker in tickers:
        path = os.path.join('fewshot_results', f'finetune_on_{from_ticker}_predict_{to_ticker}.csv')
        df = pd.read_csv(path)
        df = df.dropna(subset=["y_true", "y_pred"])
        m = metrics(df["y_true"].values, df["y_pred"].values)
        m["ticker"] = to_ticker
        results.append(m)
    fewshot_df = pd.DataFrame(results).set_index("ticker")
    output_path = os.path.join("fewshot_results", f'finetune_on_{from_ticker}.csv')
    fewshot_df.to_csv(output_path)
    print('\n\n',f'--------FINETUNE ON {from_ticker}--------')
    print(fewshot_df)


results = []
for ticker in tickers:
    path = os.path.join('fewshot_results', f'finetune_on_{ticker}_predict_{ticker}_event.csv')
    df = pd.read_csv(path)
    df = df.dropna(subset=["y_true", "y_pred"])
    m = metrics(df["y_true"].values, df["y_pred"].values)
    m["ticker"] = ticker
    results.append(m)        

    fewshot_df = pd.DataFrame(results).set_index("ticker")
    output_path = os.path.join("fewshot_results", "finetune_itself_event.csv")
    fewshot_df.to_csv(output_path)
print('\n\n','--------FINETUNE ITSELF--------')
print(fewshot_df)


for from_ticker in tickers:
    results = []
    for to_ticker in tickers:
        path = os.path.join('fewshot_results', f'finetune_on_{from_ticker}_predict_{to_ticker}_event.csv')
        df = pd.read_csv(path)
        df = df.dropna(subset=["y_true", "y_pred"])
        m = metrics(df["y_true"].values, df["y_pred"].values)
        m["ticker"] = to_ticker
        results.append(m)
    fewshot_df = pd.DataFrame(results).set_index("ticker")
    output_path = os.path.join("fewshot_results", f'finetune_on_{from_ticker}_event.csv')
    fewshot_df.to_csv(output_path)
    print('\n\n',f'--------FINETUNE ON {from_ticker}--------')
    print(fewshot_df)

Some weights of TinyTimeMixerForPrediction were not initialized from the model checkpoint at ibm-granite/granite-timeseries-ttm-r2 and are newly initialized: ['decoder.decoder_block.mixers.0.channel_feature_mixer.gating_block.attn_layer.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.gating_block.attn_layer.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc1.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc1.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc2.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc2.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.norm.norm.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.norm.norm.weight', 'decoder.decoder_block.mixers.1.channel_feature_mixer.gating_block.attn_layer.bias', 'decoder.decoder_block.mixers.1.channel_feature_mixer.gating_block.attn_layer.weight', 'decoder.decoder_block.mixers.1.channel_feature_mixer.mlp.fc1.bias', 'dec

Epoch,Training Loss,Validation Loss
1,0.914500,0.372680
2,0.508700,0.573007
3,0.396400,1.324730
4,0.341800,0.817753
5,0.324400,0.842257


[TrackingCallback] Mean Epoch Time = 40.07431750297546 seconds, Total Train Time = 390.87165880203247


Device set to use cpu
Some weights of TinyTimeMixerForPrediction were not initialized from the model checkpoint at ibm-granite/granite-timeseries-ttm-r2 and are newly initialized: ['decoder.decoder_block.mixers.0.channel_feature_mixer.gating_block.attn_layer.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.gating_block.attn_layer.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc1.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc1.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc2.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc2.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.norm.norm.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.norm.norm.weight', 'decoder.decoder_block.mixers.1.channel_feature_mixer.gating_block.attn_layer.bias', 'decoder.decoder_block.mixers.1.channel_feature_mixer.gating_block.attn_layer.weight', 'decoder.decoder_block.mixers.1.channel_feature_mix

Epoch,Training Loss,Validation Loss
1,0.626500,0.407234
2,0.489200,0.124230
3,0.401200,0.093063
4,0.368800,0.100021
5,0.358100,0.100159


...

Device set to use cpu
Some weights of TinyTimeMixerForPrediction were not initialized from the model checkpoint at ibm-granite/granite-timeseries-ttm-r2 and are newly initialized: ['decoder.decoder_block.mixers.0.channel_feature_mixer.gating_block.attn_layer.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.gating_block.attn_layer.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc1.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc1.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc2.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.mlp.fc2.weight', 'decoder.decoder_block.mixers.0.channel_feature_mixer.norm.norm.bias', 'decoder.decoder_block.mixers.0.channel_feature_mixer.norm.norm.weight', 'decoder.decoder_block.mixers.1.channel_feature_mixer.gating_block.attn_layer.bias', 'decoder.decoder_block.mixers.1.channel_feature_mixer.gating_block.attn_layer.weight', 'decoder.decoder_block.mixers.1.channel_feature_mix

Epoch,Training Loss,Validation Loss
1,0.491300,0.315975
2,0.362900,0.105022
3,0.286900,0.087877
4,0.274700,0.075727
5,0.254600,0.074843


[TrackingCallback] Mean Epoch Time = 43.680622243881224 seconds, Total Train Time = 426.82940435409546


Device set to use cpu




 --------FINETUNE ITSELF--------
              mae       rmse       mape  hit_rate  precision    recall  \
ticker                                                                   
AAPL    49.869907  59.544480  36.893818  0.475368   0.515645  0.504284   
TSLA    29.913302  38.965486  17.623667  0.485790   0.510429  0.537468   
XOM      4.026230   4.925996   6.471292  0.503519   0.507426  0.535248   
SPY     26.794812  38.949912   7.103338  0.477287   0.521739  0.558473   
JNJ      4.111497   5.747740   2.910442  0.498401   0.511860  0.529716   
AMD     12.039214  16.689758  14.833533  0.471529   0.469166  0.512550   
PG       2.436160   3.092036   1.953492  0.467051   0.501738  0.528694   

              f1  
ticker            
AAPL    0.509901  
TSLA    0.523600  
XOM     0.520966  
SPY     0.539481  
JNJ     0.520635  
AMD     0.489899  
PG      0.514863  


 --------FINETUNE ON AAPL--------
               mae        rmse       mape  hit_rate  precision    recall  \
ticker         